In [ ]:
import numpy as np
import random
from deap import base, creator, tools, algorithms
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

class SimpleNN(nn.Module):
    def __init__(self, num_neurons, activation_function):
        super(SimpleNN, self).__init__()
        self.hidden = nn.Linear(5, num_neurons)
        self.output = nn.Linear(num_neurons, 1)
        
        # Fixed indentation for activation assignment
        self.activation = {
            'relu': nn.ReLU(),
            'tanh': nn.Tanh(),
            'sigmoid': nn.Sigmoid()
        }[activation_function]

    def forward(self, x):
        x = self.activation(self.hidden(x))
        x = torch.sigmoid(self.output(x))
        return x

def objective_function(individual, X_train, y_train, X_test, y_test):
    num_neurons, learning_rate, activation_function = individual

    model = SimpleNN(num_neurons, activation_function)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=max(0.0001, learning_rate))

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

    for epoch in range(150):
        model.train()
        optimizer.zero_grad()
        predictions = model(X_train_tensor)
        loss = criterion(predictions, y_train_tensor)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        predictions = model(X_test_tensor)
        accuracy = ((predictions.round() == y_test_tensor).float().mean()).item()
    return (1-accuracy,)

def load_data():
    X = np.random.rand(500, 5)
    y = np.random.randint(0, 2, 500)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    return X_train, X_test, y_train, y_test

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

def create_individual():
    return [ 
        random.randint(10, 150),
        random.uniform(0.0001, 0.01),
        random.choice(['relu', 'tanh', 'sigmoid'])
    ]

toolbox = base.Toolbox()
toolbox.register("individual", tools.initIterate, creator.Individual, create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register('mate', tools.cxTwoPoint)

def custom_mutate(individual):
    """Fixed mutation for learning rate (removed int conversion)"""
    if random.random() < 0.2:
        individual[0] = max(10, min(150, individual[0] + int(random.gauss(0, 10))))
    
    # Corrected learning rate mutation
    if random.random() < 0.2:
        individual[1] = max(0.0001, min(0.01, individual[1] + random.gauss(0, 0.001)))
    
    if random.random() < 0.2:
        individual[2] = random.choice(['relu', 'tanh', 'sigmoid'])
    return individual,

toolbox.register("mutate", custom_mutate)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", objective_function, X_train=None, y_train=None, X_test=None, y_test=None)

def run_ga():
    X_train, X_test, y_train, y_test = load_data()
    toolbox.unregister("evaluate")
    toolbox.register("evaluate", objective_function, X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test)

    population = toolbox.population(n=25)
    algorithms.eaSimple(population, toolbox, cxpb=0.7, mutpb=0.3, ngen=20, verbose=True)

    best_individual = tools.selBest(population, 1)[0]
    print(f"Best individual: {best_individual}")
    return best_individual

best_individual = run_ga()


gen	nevals
0  	25    
1  	20    
2  	20    
3  	17    
4  	19    
5  	22    
6  	22    
7  	22    
8  	23    
9  	21    
10 	22    
11 	14    
12 	21    
13 	20    
14 	19    
